<a href="https://colab.research.google.com/github/annaluizacarvalhaes-star/Citizen-Assist-AI-SERS/blob/main/GS_CITIZEN_ASSIST_IA_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Citizen Assist AI - Módulo de Decisão Lógica (Agendamento Hospitalar SUS)

## Global Solution - Computer Science

In [1]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!sleep 3
!ollama pull llama3.2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [2]:
!ollama list
!tail -n 20 /content/ollama.log

NAME               ID              SIZE      MODIFIED      
llama3.2:latest    a80c4f17acd5    2.0 GB    6 seconds ago    
time=2026-11-20T01:40:17.123Z level=INFO source=routes.go:2474 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://loca

In [3]:
!curl -s http://127.0.0.1:11434/api/generate -d '{"model": "llama3.2", "prompt": "Responda apenas: OK", "stream": false}'

{"model":"llama3.2","created_at":"2026-11-20T01:40:47.503326804Z","response":"OK","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,1079,621,3315,47767,25,10619,128009,128006,78191,128007,271,4012],"total_duration":13111575024,"load_duration":9025373475,"prompt_eval_count":31,"prompt_eval_cached_count":0,"prompt_eval_duration":3771299000,"eval_count":2,"eval_duration":311081000}

In [4]:
import urllib.request
import urllib.error
import json

In [5]:
def analisar_consumo_energia(consumo_kwh: float, tarifa_kwh: float = 0.85) -> dict:
    # Calcula custos e compara alternativas para o período mensal.
    custo_convencional = consumo_kwh * tarifa_kwh

    # ajustar conforme regras
    custo_social = custo_convencional * (1 - 0.65)
    economia_social = custo_convencional - custo_social

    custo_solar = custo_convencional * 0.20
    economia_solar = custo_convencional - custo_solar

    return {
        "consumo_kwh": consumo_kwh,
        "tarifa_kwh": tarifa_kwh,
        "custo_convencional": custo_convencional,
        "custo_social": custo_social,
        "economia_social": economia_social,
        "custo_solar": custo_solar,
        "economia_solar": economia_solar,
    }


In [6]:
def gerar_resposta_padrao(dados: dict) -> str:
    """Resposta clara e acessível voltada a idosos e saúde pública caso o Ollama não esteja ativo."""
    return f"""
--- CITIZEN ASSIST AI | ENERGIA, IDOSOS E SAÚDE PÚBLICA ---
Público-alvo: Pessoas idosas (com foco em bem-estar e saúde pública)
Necessidade: Reduzir o peso da conta de luz sem abrir mão de iluminação segura, refrigeração e conforto térmico (período: 1 mês)

1. Situação atual (Tarifa Convencional):
   - Consumo: {dados['consumo_kwh']:.1f} kWh/mês | Tarifa base: R$ {dados['tarifa_kwh']:.2f}/kWh
   - Custo mensal estimado: R$ {dados['custo_convencional']:.2f}
   - Viabilidade e impacto ambiental: Imediata, sem cadastro adicional, porém mais cara e sem incentivo direto à geração limpa; em casas de idosos, o custo alto pode levar a cortes arriscados de iluminação ou refrigeração.

2. Alternativa 1 — Tarifa Social de Energia Elétrica (CadÚnico / BPC):
   - Custo mensal estimado (premissa de 65% de desconto): R$ {dados['custo_social']:.2f}
   - Economia mensal estimada: R$ {dados['economia_social']:.2f}
   - Viabilidade: Alta para idosos/famílias que atendem aos critérios oficiais (como BPC ou CadÚnico atualizado), exigindo inscrição e validação cadastral.
   - Impacto ambiental e na saúde: Alivia o orçamento mensal sem exigir corte de itens essenciais de saúde/segurança; funciona ainda melhor junto a lâmpadas LED e manutenção correta da geladeira.

3. Alternativa 2 — Energia Solar Fotovoltaica (Fonte Renovável):
   - Custo mensal estimado (taxa mínima / premissa de até 80% de redução): R$ {dados['custo_solar']:.2f}
   - Economia mensal estimada: R$ {dados['economia_solar']:.2f}
   - Viabilidade: Depende de investimento inicial ou financiamento, espaço e condições do telhado, além de avaliação técnica (muito relevante também em espaços comunitários ou de apoio à saúde).
   - Impacto ambiental e na saúde: Fonte renovável que reduz emissões associadas à eletricidade da rede e ajuda a estabilizar custos de energia a longo prazo.

Limitações e premissas:
- Os valores são estimativas educativas baseadas em tarifa média e percentuais simplificados; as regras reais da Tarifa Social dependem de faixas de consumo e critérios oficiais vigentes, e a geração solar depende das condições do local.
- Este assistente não é um canal oficial do governo nem substitui orientação médica, assistência social ou avaliação técnica da distribuidora.

Referências:
- ANEEL (Tarifa Social de Energia Elétrica e regulação do setor elétrico)
- CadÚnico / BPC e diretrizes de eficiência energética e saúde ambiental para pessoas idosas.
""".strip()

In [7]:
def chamar_ollama_colab(pergunta: str, dados: dict, modelo: str = "llama3.2") -> str | None:
    texto_dados = (
        f"Público: idosos; contexto: saúde pública, segurança doméstica, iluminação e conservação/conforto. "
        f"Consumo: {dados['consumo_kwh']:.1f} kWh/mês, tarifa base: R$ {dados['tarifa_kwh']:.2f}/kWh. "
        f"Convencional: R$ {dados['custo_convencional']:.2f}/mês. "
        f"Tarifa Social (premissa 65%): custo R$ {dados['custo_social']:.2f}/mês, economia R$ {dados['economia_social']:.2f}/mês. "
        f"Solar fotovoltaico (premissa até 80% redução): custo R$ {dados['custo_solar']:.2f}/mês, economia R$ {dados['economia_solar']:.2f}/mês."
    )

    prompt = (
        "Você é o Citizen Assist AI no módulo de Energia e Sustentabilidade, voltado a pessoas idosas e saúde pública. "
        "Use linguagem simples, respeitosa, legível e acolhedora. "
        "Relacione o uso consciente de energia ao cuidado com a saúde e segurança do idoso (como boa iluminação e refrigeração adequada, sem cortar o que é essencial à saúde). "
        "Use apenas os números informados, sem inventar valores. "
        "Compare custo, viabilidade e impacto ambiental das alternativas (incluindo fonte renovável), "
        "deixe claras as limitações/premissas (sem tratar estimativas como garantia ou canal oficial) e cite referências.\n\n"
        f"Dados: {texto_dados}\n"
        f"Dúvida do usuário: {pergunta}"
    )

    payload = {
        "model": modelo,
        "prompt": prompt,
        "stream": False,
    }

    try:
        req = urllib.request.Request(
            "http://127.0.0.1:11434/api/generate",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=120) as resp:
            resultado = json.loads(resp.read().decode("utf-8"))
            return resultado.get("response", "").strip() or None
    except Exception as e:
        print(f"[DEBUG Ollama] Falha ao chamar o modelo '{modelo}': {type(e).__name__}: {e}")
        return None

In [ ]:
def assistente_citizen_assist():
    print("=== Citizen Assist AI — Energia, Idosos e Saúde Pública ===")
    print("1 - Simular consumo e comparar alternativas (Tarifa Social vs. Solar)")
    print("2 - Sair\n")

    while True:
        escolha = input("Digite 1 ou 2: ").strip()

        if escolha == "2":
            print("Encerrando o assistente. Cuide-se bem!")
            break

        if escolha != "1":
            print("Opção inválida. Digite 1 ou 2.\n")
            continue

        try:
            consumo = float(input("Informe o consumo mensal médio em kWh (ex.: 150): ").replace(",", "."))
            if consumo <= 0:
                print("Digite um consumo maior que zero.\n")
                continue
        except ValueError:
            print("Valor inválido. Digite um número, por exemplo 150.\n")
            continue

        pergunta = input(
            "Qual é a sua dúvida? (ex.: como economizar sem prejudicar a saúde ou manter a geladeira?): "
        ).strip()
        if not pergunta:
            pergunta = "Como posso economizar na conta de luz cuidando da saúde e segurança de um idoso?"

        dados = analisar_consumo_energia(consumo)
        resposta_ia = chamar_ollama_colab(pergunta, dados)

        print("\n" + "-" * 60)
        if resposta_ia:
            print(resposta_ia)
        else:
            print("(Ollama não disponível no momento — exibindo orientação padrão)\n")
            print(gerar_resposta_padrao(dados))
        print("-" * 60 + "\n")


assistente_citizen_assist()

=== Citizen Assist AI — Energia, Idosos e Saúde Pública ===
1 - Simular consumo e comparar alternativas (Tarifa Social vs. Solar)
2 - Sair

Digite 1 ou 2: 1
Informe o consumo mensal médio em kWh (ex.: 150): como economizar luz
Valor inválido. Digite um número, por exemplo 150.

Digite 1 ou 2: 1
Informe o consumo mensal médio em kWh (ex.: 150): 100
Qual é a sua dúvida? (ex.: como economizar sem prejudicar a saúde ou manter a geladeira?): como economizar
[DEBUG Ollama] Falha ao chamar o modelo 'llama3.2': TimeoutError: timed out

------------------------------------------------------------
(Ollama não disponível no momento — exibindo orientação padrão)

--- CITIZEN ASSIST AI | ENERGIA, IDOSOS E SAÚDE PÚBLICA ---
Público-alvo: Pessoas idosas (com foco em bem-estar e saúde pública)
Necessidade: Reduzir o peso da conta de luz sem abrir mão de iluminação segura, refrigeração e conforto térmico (período: 1 mês)

1. Situação atual (Tarifa Convencional):
   - Consumo: 100.0 kWh/mês | Tarifa bas